# 02 · Schema Classification — which columns become features, and how

Stage 3 of the pipeline (`scripts/classify_schema.py`). Between the split and the tokenizer, this
step reads the **train** panel and decides, for every column: is it a **feature** or a **drop**? is
it **static** (a fixed origination fact → the *profile* branch) or **dynamic** (changes every month
→ the *event* branch)? and is its value **numeric**, **categorical**, or a **flag**? The output is
the **field schema** the tokenizer then turns into tokens.

It runs on the **train split only** so any cardinality statistics stay train-only (leakage rule
DL-008) — the same discipline as fitting the vocabulary.

**Contents**
1. What this stage does
2. Role — static (profile) vs dynamic (event)
3. Type — numeric / categorical / flag / temporal / constant
4. Dropping — constant &amp; redundant (safe) vs functional-dependency (review)
5. The resulting Fannie field schema
6. How to run it
7. Notes &amp; caveats

## Setup

In [ ]:
from pathlib import Path

import pandas as pd
import yaml

# find the repo root (walk up until we see configs/)
ROOT = Path.cwd()
while not (ROOT / "configs" / "fannie_mae").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "configs" / "fannie_mae").exists(), "run inside the credit-foundation-model repo"

def load_yaml(name):
    return yaml.safe_load((ROOT / "configs" / "fannie_mae" / name).read_text())

CLASSIFY = load_yaml("classify.yaml")     # the recipe (input = train split, id/time cols)
SCHEMA = load_yaml("tokenizer.yaml")      # the OUTPUT field schema this stage feeds
BASE = load_yaml("baseline.yaml")         # leakage / exclude lists (context)
print("classify recipe input:", CLASSIFY.get("input"))
print("id_col / time_col     :", CLASSIFY.get("id_col"), "/", CLASSIFY.get("time_col"))

## 1. What this stage does

`classify_schema.py` is **data-driven and reproducible** — re-running it on the same train split
regenerates the same schema. Five steps:

1. **Classify each column's role** — `id` / `static` (constant within a loan) / `dynamic` (varies
   across a loan's monthly rows).
2. **Classify each column's value type** — `constant` / `temporal` / `flag` / `bucket` / `numeric`
   / `categorical` / `text`.
3. **Drop `constant` columns** — one value across the whole panel = no signal.
4. **Drop `safe` redundancies** auto-detected from the data — exact-duplicate columns and numeric
   `*_bucket` pre-discretizations (the raw is kept and re-binned by the tokenizer).
5. **Emit the profile (static) / event (dynamic) field lists**, each split by type — this is the
   schema the tokenizer consumes.

Functional-dependency candidates (`X = f(Y)`) are **not** auto-dropped — they're printed as
`review` suggestions, because some (like an explicit flag) may be wanted as signals. You opt in via
the recipe's `drop:` list.

## 2. Role — static (profile) vs dynamic (event)

The test is simple and structural: **group the panel by `loan_id`; if a column has one value within
every loan, it's static; if it changes across a loan's months, it's dynamic.** (Computed on a random
sample of loans for speed — it's a structural property, so a sample is enough.)

| Role | Meaning | Branch | Fannie examples |
|---|---|---|---|
| **static** | fixed at origination, repeats every month | **PROFILE** (emitted once per loan) | `original_ltv`, `dti`, `fico`, `loan_purpose`, `property_state` |
| **dynamic** | changes month to month | **EVENT** (emitted per monthly row) | `current_interest_rate`, `current_actual_upb`, `loan_age`, `remaining_months_to_maturity` |
| **id** | the loan key | — | `loan_id` |

This static/dynamic split is exactly what the three-branch model needs: the Profile encoder reads the
static block once, the Event encoder reads each month's dynamic block.

## 3. Type — numeric / categorical / flag / temporal / constant

Type is decided by name and cardinality (number of distinct values):

| Type | Rule | Handling downstream |
|---|---|---|
| `constant` | only 1 distinct value | **dropped** (no signal) |
| `temporal` | name ends `_date`, or is the time column | becomes the `t=` / `cal=` time coordinates |
| `flag` | name ends `_flag` | a small yes/no/NA categorical |
| `bucket` | name ends `_bucket` | pre-binned (dropped if it just discretizes a numeric raw) |
| `numeric` | numeric dtype with **> 20** distinct values | quantile-binned by the tokenizer |
| `categorical` | numeric with **≤ 20** distinct, or a low-cardinality string | one token per category |
| `text` | very high-cardinality / long strings | excluded (non-tabular) |

The `> 20` cutoff is why something like `number_of_units` (values 1–4) is treated as **categorical**
while `original_upb` (thousands of distinct dollar amounts) is **numeric** and gets binned.

## 4. Dropping — `safe` (auto) vs `review` (human)

The classifier separates two kinds of redundancy so nothing is silently lost:

* **`safe` — auto-dropped.** (a) **Exact duplicates** — two columns with identical values on the
  sample (keep one). (b) **Numeric `*_bucket`** — a pre-discretized copy of a numeric raw column;
  the raw is kept and the tokenizer re-bins it. *(A `*_bucket` with no numeric base — e.g. a genuine
  state code — is a real field and is NOT dropped.)*
* **`review` — surfaced, not dropped.** **Functional dependencies** `X = f(Y)` among low-cardinality
  columns (keep the finer `Y`). These print as suggestions; you drop them explicitly via the recipe's
  `drop:` list only if you're sure they carry no independent signal.

This "auto-drop only the unambiguous, ask about the rest" split is the whole point — reproducible
where it's safe, human-in-the-loop where judgment matters.

## 5. The resulting Fannie field schema

Below is the field schema this stage feeds the tokenizer (`configs/fannie_mae/tokenizer.yaml`):
every kept field, its branch (profile/event) and type (numeric/categorical). Note the counts — the
tokenizer works from a **curated ~43-field** subset of the 57 no-leakage features (see the honesty
note after the table).

In [ ]:
def schema_rows(schema):
    rows = []
    for branch in ("profile", "event"):
        for typ, cols in (schema.get(branch) or {}).items():
            for c in (cols or []):
                rows.append({"field": c, "branch": branch, "type": typ})
    return pd.DataFrame(rows)

FIELDS = schema_rows(SCHEMA)
counts = FIELDS.groupby(["branch", "type"]).size().unstack(fill_value=0)
print(f"{len(FIELDS)} tokenized fields")
display(counts)
FIELDS

In [ ]:
import matplotlib.pyplot as plt
pivot = FIELDS.groupby(["branch", "type"]).size().unstack(fill_value=0)
ax = pivot.plot(kind="bar", figsize=(8, 4))
ax.set_title("Fannie field schema — fields per branch × type")
ax.set_xlabel("branch")
ax.set_ylabel("fields")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

### ⚠️ Honesty note — for Fannie this schema is *curated*, not auto-generated

`classify_schema.py` is the **automated classifier**, and it is the generator of record for the
**Dutch** validation panel. For **Fannie** the `tokenizer.yaml` above was **hand-curated** on top of
the classifier's suggestions, for three reasons the auto-classifier doesn't handle:

* **Leakage.** `classify_schema` does not read `baseline.yaml`'s leakage list, so it would happily
  route `current_loan_delinquency_status`, `zero_balance_code`, etc. into the model. The curated
  schema **excludes all leakage columns** — this is where "57 features, not 118" is enforced.
* **ARM/IO fields** that are null for a fixed-rate book are dropped (no signal, just noise).
* **Regulatory bin anchors** (LTV 80/90/95/97, DTI 36/43/45) — a tokenizer detail set deliberately.

So treat `classify_schema` as the reproducible **first-pass suggester + Dutch generator**; the Fannie
field schema is a reviewed artifact. (A good follow-up would be to teach `classify_schema` the leakage
list so the Fannie schema could regenerate too — tracked as a cleanup, not a blocker.)

In [ ]:
# confirm no leakage column leaked into the schema
leak = set(BASE.get("leakage", [])) | set(BASE.get("exclude", []))
bad = [c for c in FIELDS["field"] if c in leak]
print("leakage/exclude columns present in the field schema:", bad or "none ✓")

## 6. How to run it

Report-only (prints the classification + `safe`/`review` suggestions, writes nothing):

```bash
python scripts/classify_schema.py -c configs/fannie_mae/classify.yaml
```

Generate a schema file (Dutch panel, or a Fannie first-pass to review):

```bash
python scripts/classify_schema.py -c configs/fannie_mae/classify.yaml \
    --out configs/fannie_mae/tokenizer.gen.yaml --drop '[some_redundant_col]'
```

It reads `${paths.processed}/train.parquet` — the **train split only**, so cardinality/redundancy
stats never see val/test (DL-008). Its unit tests live in `tests/test_data.py`
(`test_find_redundant_flags_duplicate_and_fd`).

## 7. Notes &amp; caveats

* **Train-only.** Runs on `train.parquet` so the classification can't peek at val/test — same leakage
  discipline as the vocabulary fit.
* **This stage decides *fields*, the next fits *values*.** Classification says "`original_ltv` is a
  numeric profile field"; the **tokenizer** (stage 4) then learns *how* to bin it (16 bins, anchored
  at 80/90/95/97) on the train data. Kept separate on purpose.
* **Static/dynamic drives the architecture.** The profile/event split here is exactly what the
  three-branch model consumes — profile fields feed the Profile encoder (once), event fields feed the
  Event encoder (per month).
* **Curated ≠ hand-waved.** The Fannie schema is reviewed, but every field in it is defensible and
  leakage-free (verified in the cell above). The next notebook (`03`) fits the KVT vocabulary on this
  schema and shows a loan turned into tokens.